# Imports

In [ ]:
import zarr
import numpy as np
from zarr.storage import LMDBStore
from sklearn.metrics import accuracy_score

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import confusion_matrix

from sklearn.decomposition import PCA
from scipy.stats import entropy
from sklearn.metrics import pairwise_distances

In [ ]:
zarr_preds_path = '../outputs/predictions_full_augmentation.lmdb'
aug_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
aug_store = zarr.open_group(aug_store, mode="r")

In [ ]:
zarr_preds_path = '../outputs/predictions_noise_augmentation.lmdb'
noise_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
noise_store = zarr.open_group(noise_store, mode="r")

# Functions for reading and compiling

In [ ]:
def load_image_outputs(
    store: zarr.Group,
    trial_id: str,
    image_name: str,
    load_top_k: bool = False,
) -> dict:
    grp = store[f"{trial_id}/{image_name}"]

    result = {
        "argmax_map": grp["argmax_map"][:],
        "bg_prob":    grp["bg_prob"][:],
        "attrs":      dict(grp.attrs),
    }

    if load_top_k:
        result["top_k_classes"] = grp["top_k_classes"][:]
        result["top_k_probs"]   = grp["top_k_probs"][:]

    return result

def compile_results(store: zarr.Group) -> dict:
    trial_list = sorted(store.keys())
    result_dict = {}
    
    for trial_id in trial_list:
        image_names = sorted(store[trial_id].keys())
        for image_name in image_names:
            img_result = load_image_outputs(store, trial_id, image_name)
            samples_per_class = img_result['attrs']['N']
    
            if samples_per_class not in result_dict:
                result_dict[samples_per_class] = {}
    
            if trial_id not in result_dict[samples_per_class]:
                result_dict[samples_per_class][trial_id] = {'gt': [], 'pred': []}
    
            preds = img_result['argmax_map']
            true_idx = img_result['attrs']['true_idx']
            nonzero_preds = preds[preds != 0]
            gt = np.ones(nonzero_preds.shape) * true_idx
    
            result_dict[samples_per_class][trial_id]['gt'].append(gt)
            result_dict[samples_per_class][trial_id]['pred'].append(nonzero_preds)
    
    for samples_per_class in result_dict:
        for trial_id in result_dict[samples_per_class]:
            result_dict[samples_per_class][trial_id]['gt'] = np.concatenate(result_dict[samples_per_class][trial_id]['gt'])
            result_dict[samples_per_class][trial_id]['pred'] = np.concatenate(result_dict[samples_per_class][trial_id]['pred'])

    return result_dict

def get_accuracy(result_dict: dict) -> tuple:
    sizes = list(result_dict.keys())
    
    perf_list = []
    for _, trials in result_dict.items():
        accs = [accuracy_score(t['gt'], t['pred']) for t in trials.values() if t]
        perf_list.append(accs)
    
    acc = np.array([[np.mean(a), np.std(a)] for a in perf_list]).T
    return sizes, acc

In [ ]:
def load_image_outputs(
    store: zarr.Group,
    trial_id: str,
    image_name: str,
    load_top_k: bool = False,
) -> dict:
    grp = store[f"{trial_id}/{image_name}"]
    layout = grp.attrs.get("layout", "spatial")  # default to spatial format

    if layout == "flat":
        result = {
            "argmax":      grp["argmax"][:],
            "bg_prob":     grp["bg_prob"][:],
            "true_labels": grp["true_labels"][:] if "true_labels" in grp else None,
            "attrs":       dict(grp.attrs),
            "layout":      "flat",
        }
    else:
        result = {
            "argmax_map": grp["argmax_map"][:],
            "bg_prob":    grp["bg_prob"][:],
            "attrs":      dict(grp.attrs),
            "layout":     "spatial",
        }

    if load_top_k:
        result["top_k_classes"] = grp["top_k_classes"][:]
        result["top_k_probs"]   = grp["top_k_probs"][:]
    return result


def compile_results(
    store: zarr.Group,
    augment_train: bool | None = None,
    seeds: list[int] | None = None,
    spectra_per_class: list[int] | None = None,
    image_names: list[str] | None = None,
) -> dict:
    """
    Compile GT/prediction arrays across trials, grouped by samples-per-class (N).
    Args:
        store:               the LMDB/zarr prediction store.
        augment_train:       if True/False, only include trials where
                             attrs["aug"]["augment_train"] matches this value.
                             If None (default), include all trials regardless
                             of augmentation status.
        seeds:               if provided, only include trials where
                             attrs["seed"] is in this list.
                             e.g. seeds=[5, 8]
        spectra_per_class:   if provided, only include trials where
                             attrs["spectra_per_class"] is in this list.
                             e.g. spectra_per_class=[32, 512]
    """
    trial_list = sorted(store.keys())
    result_dict: dict = {}
    _seeds = {int(s) for s in seeds} if seeds is not None else None
    _spectra_per_class = {int(s) for s in spectra_per_class} if spectra_per_class is not None else None
    
    for trial_id in trial_list:
        
        if image_names is not None:
            image_names_to_use = [f for f in sorted(store[trial_id].keys()) if f in image_names]
        else:
            image_names_to_use = sorted(store[trial_id].keys()) 
        
        for image_name in image_names_to_use:
            img_result = load_image_outputs(store, trial_id, image_name)
            attrs = img_result["attrs"]

            # ── filter by augmentation flag if requested ────────────────────
            if augment_train is not None:
                aug_info = attrs.get("aug", {})
                trial_augment_train = aug_info.get("augment_train", False)
                if bool(trial_augment_train) != augment_train:
                    continue

            # ── filter by seed if requested ─────────────────────────────────
            if _seeds is not None:
                if attrs.get("seed") not in _seeds:
                    continue

            # ── filter by spectra_per_class if requested ────────────────────
            if _spectra_per_class is not None:
                if attrs.get("N") not in _spectra_per_class:
                    continue

            samples_per_class = attrs["N"]
            if samples_per_class not in result_dict:
                result_dict[samples_per_class] = {}
            if trial_id not in result_dict[samples_per_class]:
                result_dict[samples_per_class][trial_id] = {"gt": [], "pred": []}
            if img_result["layout"] == "flat":
                # PCUK / milk — per-pixel/per-sample GT already stored
                preds = img_result["argmax"]
                gt    = img_result["true_labels"]
                if gt is None:
                    # fall back to single true_idx if true_labels wasn't saved
                    true_idx = attrs["true_idx"]
                    gt = np.full(preds.shape, true_idx)
                # no background filtering — every entry is a real labelled sample
                result_dict[samples_per_class][trial_id]["gt"].append(gt)
                result_dict[samples_per_class][trial_id]["pred"].append(preds)
            else:
                # legacy spatial — single label per image, background_idx=0
                preds = np.asarray(img_result["argmax_map"]).reshape(-1)
                true_idx = attrs["true_idx"]
                
                if np.ndim(true_idx) == 0:
                    nonzero_preds = preds[preds != 0]
                    gt = np.full(nonzero_preds.shape, true_idx)
                
                    result_dict[samples_per_class][trial_id]["gt"].append(gt)
                    result_dict[samples_per_class][trial_id]["pred"].append(nonzero_preds)
                
                else:
                    true_idx = np.asarray(true_idx).reshape(-1)
                
                    if true_idx.shape != preds.shape:
                        raise ValueError(
                            f"Shape mismatch: preds {preds.shape}, true_idx {true_idx.shape}"
                        )
                
                    mask = preds != np.nan
                    gt = true_idx[mask]
                    pred = preds[mask]
                
                    result_dict[samples_per_class][trial_id]["gt"].append(gt)
                    result_dict[samples_per_class][trial_id]["pred"].append(pred)


    for samples_per_class in result_dict:
        for trial_id in result_dict[samples_per_class]:
            entry = result_dict[samples_per_class][trial_id]
            if entry["gt"]:
                entry["gt"]   = np.concatenate(entry["gt"])
                entry["pred"] = np.concatenate(entry["pred"])
    return result_dict

def get_accuracy(result_dict: dict) -> tuple:
    sizes = list(result_dict.keys())

    perf_list = []
    for _, trials in result_dict.items():
        accs = [
            accuracy_score(t["gt"], t["pred"])
            for t in trials.values()
            if len(t.get("gt", [])) > 0
        ]
        perf_list.append(accs)

    acc = np.array([[np.mean(a) if a else np.nan, np.std(a) if a else np.nan] for a in perf_list]).T
    return sizes, acc


# Statistics

In [ ]:
images_group = root["images"]
ALL_CLASSES = list(root.attrs["classes"])[:4]

data = {cls: [] for cls in ALL_CLASSES}

for name in sorted(images_group.keys()):
    y = images_group[name]["y"][:]
    for i, cls in enumerate(ALL_CLASSES):
        data[cls].append(int(np.sum(y == i)))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Pixel count per core — first 4 classes", fontsize=14)

for ax, cls in zip(axes.flat, ALL_CLASSES):
    counts = data[cls]
    ax.hist(counts, bins=100, edgecolor="black")
    ax.set_title(cls)
    ax.set_xlabel("Pixel count")
    ax.set_ylabel("Number of cores")
    ax.axvline(np.median(counts), color="red", linestyle="--", label=f"Median: {np.median(counts):.0f}")
    ax.legend()


In [ ]:
import zarr
import numpy as np

zarr_path = "/mnt/ssd3/eirik/ProcessedData/pcuk.zarr"

root = zarr.open(zarr_path, mode="r")
images_group = root["images"]
ALL_CLASSES = list(root.attrs["classes"])

print(f"{'Core':<40} " + " ".join(f"{c[:8]:>10}" for c in ALL_CLASSES))
print("-" * (40 + 11 * len(ALL_CLASSES)))

for name in sorted(images_group.keys()):
    y = images_group[name]["y"][:]
    counts = [int(np.sum(y == i)) for i in range(len(ALL_CLASSES))]
    print(f"{name:<40} " + " ".join(f"{c:>10,}" for c in counts))

# Read data

## PCUK

In [ ]:
zarr_preds_path = '../outputs/predictions_pcuk.lmdb'
store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
store = zarr.open_group(store, mode="r")

In [ ]:
from joblib import dump, load
import pickle

seed_list = list(range(1, 4))
spectra_per_class_list = [2**i for i in range(5, 16)]
spectra_per_class_list = [2**i for i in range(10, 16)]

name = 'pcuk'

result_dict = compile_results(store, augment_train=False, 
                              # spectra_per_class=spectra_per_class_list
                             )
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(store, augment_train=True, 
                              # spectra_per_class=spectra_per_class_list
                             )
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

store.store.close()

## Microplastics

In [ ]:
zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_raw.lmdb'
raw_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
raw_store = zarr.open_group(raw_store, mode="r")

zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_full_augmentation.lmdb'
aug_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
aug_store = zarr.open_group(aug_store, mode="r")

In [ ]:
seed_list = list(range(1, 4))
spectra_per_class_list = [2**i for i in range(5, 16)]
spectra_per_class_list = [2**i for i in range(10, 16)]

name = 'microplastics'

result_dict = compile_results(raw_store)
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(aug_store)
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

aug_store.store.close()
raw_store.store.close()

## Milk

In [ ]:
zarr_preds_path = '../outputs/predictions_milk.lmdb'
store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
store = zarr.open_group(store, mode="r")

In [ ]:
from joblib import dump, load
import pickle

seed_list = list(range(1, 4))
spectra_per_class_list = [2**i for i in range(5, 16)]
spectra_per_class_list = [2**i for i in range(10, 16)]

name = 'milk'

result_dict = compile_results(store, augment_train=False, 
                              # spectra_per_class=spectra_per_class_list
                             )
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(store, augment_train=True, 
                              # spectra_per_class=spectra_per_class_list
                             )
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

store.store.close()

## MLROD

In [ ]:
zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_mlrod_raw.lmdb'
raw_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
raw_store = zarr.open_group(raw_store, mode="r")

zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_mlrod_full_augmentation.lmdb'
aug_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
aug_store = zarr.open_group(aug_store, mode="r")

In [ ]:
seed_list = list(range(1, 4))
spectra_per_class_list = [2**i for i in range(5, 14)]

name = 'mlrod'

result_dict = compile_results(raw_store, spectra_per_class=spectra_per_class_list)
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(aug_store, spectra_per_class=spectra_per_class_list)
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

aug_store.store.close()
raw_store.store.close()

## Bacteria

In [ ]:
zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_bacteria.lmdb'
store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
store = zarr.open_group(store, mode="r")

In [ ]:
for trial_id in sorted(store.keys()):
    for image_name in sorted(store[trial_id].keys()):
        ti = store[f"{trial_id}/{image_name}"].attrs.get("true_idx")
        arr = np.asarray(ti)
        if arr.ndim > 0:
            print(trial_id, image_name, "true_idx shape:", arr.shape, "layout:", store[f"{trial_id}/{image_name}"].attrs.get("layout"))

In [ ]:
image_names = ['ho_2018clinical', 'ho_2019clinical']

name = 'bacteria_clinical'
result_dict = compile_results(store, augment_train=False, 
                              # spectra_per_class=spectra_per_class_list
                              image_names = image_names
                             )
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(store, augment_train=True, 
                              # spectra_per_class=spectra_per_class_list
                              image_names = image_names
                             )
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
store.store.close()

## Pollen

In [ ]:
zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_pollen_raw.lmdb'
raw_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
raw_store = zarr.open_group(raw_store, mode="r")

zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_pollen_full_augmentation.lmdb'
aug_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
aug_store = zarr.open_group(aug_store, mode="r")

In [ ]:
seed_list = list(range(1, 4))
spectra_per_class_list = [2**i for i in range(5, 14)]

name = 'pollen'

result_dict = compile_results(raw_store, spectra_per_class=spectra_per_class_list)
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(aug_store, spectra_per_class=spectra_per_class_list)
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

aug_store.store.close()
raw_store.store.close()

In [ ]:
sorted(raw_sizes), sorted(aug_sizes)

## Textiles

In [ ]:
zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_binary_textile_raw.lmdb'
raw_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
raw_store = zarr.open_group(raw_store, mode="r")

zarr_preds_path = '/mnt/ssd2/eirik/outputs/predictions_binary_textile_full_augmentation.lmdb'
aug_store = LMDBStore(zarr_preds_path, 
                  map_size=int(1e12), readonly=True)
aug_store = zarr.open_group(aug_store, mode="r")

In [ ]:
name = 'textile'

result_dict = compile_results(raw_store)
dump(result_dict, f"../outputs/processed_results/raw_{name}_results_dict.joblib")
raw_sizes, raw_acc = get_accuracy(result_dict)

result_dict = compile_results(aug_store)
dump(result_dict, f"../outputs/processed_results/aug_{name}_results_dict.joblib")
aug_sizes, aug_acc = get_accuracy(result_dict)

data = {
    "raw_sizes": raw_sizes,
    "raw_acc": raw_acc,
    "aug_sizes": aug_sizes,
    "aug_acc": aug_acc,
}

with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "wb") as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

aug_store.store.close()
raw_store.store.close()

# Performance vs Dataset Size

## Plot one

In [ ]:
idx = np.argsort(raw_sizes)
plt.plot(np.array(raw_sizes)[idx], raw_acc[0, idx],  label='Raw spectra', 
            alpha=0.8, lw=2., color='tab:orange')
plt.fill_between(np.array(raw_sizes)[idx], raw_acc[0, idx] - raw_acc[1, idx], raw_acc[0, idx] + raw_acc[1, idx], raw_acc[0, idx],  
            alpha=0.2, color='tab:orange', edgecolor='white')

idx = np.argsort(aug_sizes)
plt.plot(np.array(aug_sizes)[idx], aug_acc[0, idx],  label='Augmented spectra', 
            alpha=0.8, lw=2., color='tab:blue')
plt.fill_between(np.array(aug_sizes)[idx], aug_acc[0, idx] - aug_acc[1, idx], aug_acc[0, idx] + aug_acc[1, idx], aug_acc[0, idx],  
            alpha=0.2, color='tab:blue', edgecolor='white')

plt.gca().set_xscale('log')
plt.xlabel('Spectra per class in training data', fontsize=14, labelpad=10)
plt.ylabel('Classifier perfomance', fontsize=14, labelpad=15)
plt.legend(handletextpad=0.5, frameon=0, fontsize=12, loc='lower right')

ax = plt.gca()
ax.tick_params(labelsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
# plt.savefig('raw_spectra_acc_v_spectra_per_class.svg', format='svg', bbox_inches='tight', dpi=300)

## Plot all

In [ ]:
for name in ['textile', 'microplastics', 'pcuk', 'pollen', 'milk', 'bacteria_test', 'mlrod']:

    with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "rb") as f:
        data = pickle.load(f)
    
    raw_sizes = data["raw_sizes"]
    raw_acc = data["raw_acc"]
    aug_sizes = data["aug_sizes"]
    aug_acc = data["aug_acc"]

    idx = np.argsort(raw_sizes)
    plt.plot(np.array(raw_sizes)[idx], raw_acc[0, idx],  label='Raw spectra', 
                alpha=0.8, lw=2., color='tab:orange')
    plt.fill_between(np.array(raw_sizes)[idx], raw_acc[0, idx] - raw_acc[1], raw_acc[0, idx] + raw_acc[1], raw_acc[0, idx],  
                alpha=0.2, color='tab:orange', edgecolor='white')
    
    idx = np.argsort(aug_sizes)
    plt.plot(np.array(aug_sizes)[idx], aug_acc[0, idx],  label='Augmented spectra', 
                alpha=0.8, lw=2., color='tab:blue')
    plt.fill_between(np.array(aug_sizes)[idx], aug_acc[0, idx] - aug_acc[1], aug_acc[0, idx] + aug_acc[1], aug_acc[0, idx],  
                alpha=0.2, color='tab:blue', edgecolor='white')
    
    plt.gca().set_xscale('log')
    plt.xlabel('Spectra per class in training data', fontsize=14, labelpad=10)
    plt.ylabel('Classifier perfomance', fontsize=14, labelpad=15)
    plt.legend(handletextpad=0.5, frameon=0, fontsize=12, loc='lower right')
    plt.title(name)
    ax = plt.gca()
    ax.tick_params(labelsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

## Plot Delta

In [ ]:
import re
import numpy as np
from sklearn.metrics import accuracy_score

def _seed_key(tid):
    m = re.search(r'seed\d+', tid)
    return m.group(0) if m else tid

def get_delta(raw_dict: dict, aug_dict: dict, return_counts: bool = False):
    """
    Paired augmentation gain per N, matching raw/aug by seed token (seedNN)
    so that differing trial_id suffixes (e.g. '_raw' vs '_noise+...') still pair.
    """
    sizes = sorted(set(raw_dict) & set(aug_dict))

    means, stds, ns_paired = [], [], []
    for N in sizes:
        # remap this N's trials by seed token
        raw_by_seed = {_seed_key(k): v for k, v in raw_dict[N].items()}
        aug_by_seed = {_seed_key(k): v for k, v in aug_dict[N].items()}

        shared = sorted(set(raw_by_seed) & set(aug_by_seed))

        deltas = []
        for s in shared:
            r, a = raw_by_seed[s], aug_by_seed[s]
            if not r or not a:
                continue
            acc_raw = accuracy_score(r['gt'], r['pred'])
            acc_aug = accuracy_score(a['gt'], a['pred'])
            deltas.append(acc_aug - acc_raw)

        deltas = np.array(deltas)
        means.append(deltas.mean() if deltas.size else np.nan)
        stds.append(deltas.std(ddof=1) if deltas.size > 1 else np.nan)
        ns_paired.append(deltas.size)

    delta = np.array([means, stds])
    if return_counts:
        return sizes, delta, np.array(ns_paired)
    return sizes, delta

In [ ]:
names = ['textile', 'microplastics', 'pollen', 'milk', 'bacteria_test', 'mlrod']
colors = plt.cm.tab20(np.linspace(0, 1, 20)) 

for i, name in enumerate(names):
    fig, ax = plt.subplots(figsize=(7, 5))
    raw = load(f"../outputs/processed_results/raw_{name}_results_dict.joblib")
    aug = load(f"../outputs/processed_results/aug_{name}_results_dict.joblib")
    sizes, (delta_mean, delta_std) = get_delta(raw, aug)

    c = colors[i]
    ax.plot(sizes, delta_mean, label=name, alpha=0.9, lw=2., color=c, zorder=3)
    ax.fill_between(sizes, delta_mean - delta_std, delta_mean + delta_std,
                    alpha=0.2, color=c, edgecolor='white', zorder=2)

    # ax.axhline(0, color='black', lw=0.8, alpha=0.6, zorder=1)
    
    ax.set_xscale('log')               
    ymin, ymax = ax.get_ylim()
    ax.axhspan(ymin, 0, color='gray', alpha=0.15, zorder=0, lw=0)
    ax.set_ylim(ymin, ymax)           
    ax.set_xlabel('Spectra per class in training data', fontsize=14, labelpad=10)
    ax.set_ylabel(r'$\Delta$ Accuracy (aug $-$ raw)', fontsize=14, labelpad=15)
    ax.legend(handletextpad=0.5, frameon=0, fontsize=11, loc='best', ncol=2)
    
    ax.tick_params(labelsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

# Scaling laws

In [ ]:
idx = np.argsort(raw_sizes)
plt.plot(np.array(raw_sizes)[idx], 1 - raw_acc[0, idx],  label='Raw spectra', 
            alpha=0.8, lw=2., color='tab:orange')

idx = np.argsort(aug_sizes)
plt.plot(np.array(aug_sizes)[idx], 1 - aug_acc[0, idx],  label='Augmented spectra', 
            alpha=0.8, lw=2., color='tab:blue')

plt.gca().set_xscale('log')
plt.xlabel('Spectra per class in training data', fontsize=14, labelpad=10)
plt.ylabel('1 - Accuracy', fontsize=14, labelpad=15)
plt.legend(handletextpad=0.5, frameon=0, fontsize=12, loc='upper right')

ax = plt.gca()
ax.tick_params(labelsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

In [ ]:
def scaling_law(N, a, alpha, b):
    return a * N**(-alpha) + b

idx = np.argsort(raw_sizes)

cut = 0
N_list = np.array(raw_sizes)[idx][cut:]
raw_error = 1 - raw_acc[0, idx][cut:]
aug_error = 1 - aug_acc[0, idx][cut:]

raw_std = raw_acc[1, idx][cut:] + 1e-5
aug_std = aug_acc[1, idx][cut:] + 1e-5

In [ ]:
raw_params, raw_covariance = curve_fit(
    scaling_law,
    N_list,
    raw_error,
    sigma=raw_std,
    absolute_sigma=True,
    p0=[0.5, 0.3, 0.05],
    bounds=([0, 0, 0], [np.inf, 5, 1])
)

aug_params, aug_covariance = curve_fit(
    scaling_law,
    N_list,
    aug_error,
    sigma=aug_std,
    absolute_sigma=True,
    p0=[0.5, 0.3, 0.05],
    bounds=([0, 0, 0], [np.inf, 5, 1])
)


# Extract parameters
a_raw, alpha_raw, b_raw = raw_params
a_aug, alpha_aug, b_aug = aug_params

# Plot data
plt.plot(
    N_list, raw_error,
    'o', color='tab:orange', label="Raw data"
)

plt.plot(
    N_list, scaling_law(N_list, *raw_params),
    '--', color='tab:orange',
    label=(
        rf"Fit: $a={a_raw:.2f}$, "
        rf"$\alpha={alpha_raw:.2f}$, "
        rf"$b={b_raw:.3f}$"
    )
)

plt.plot(
    N_list, aug_error,
    'o', color='tab:blue', label="Augmented data"
)

plt.plot(
    N_list, scaling_law(N_list, *aug_params),
    '--', color='tab:blue',
    label=(
        rf"Fit: $a={a_aug:.2f}$ , "
        rf"$\alpha={alpha_aug:.2f}$, "
        rf"$b={b_aug:.3f}$"
    )
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Number of training samples per class $N$")
plt.ylabel("Error ($1-$Accuracy)")
plt.title('PCUK')
plt.legend(fontsize=10)

In [ ]:
N0 = max(raw_sizes)
raw_normalized_error = (scaling_law(N_list, *raw_params) - b_raw)/(scaling_law([N0], *raw_params) - b_raw)
aug_normalized_error = (scaling_law(N_list, *aug_params) - b_aug)/(scaling_law([N0], *aug_params) - b_aug)

plt.plot(N_list / N0, raw_normalized_error)
plt.plot(N_list / N0, aug_normalized_error)

plt.xscale("log")
plt.yscale("log")

# Universality

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def scaling_law(N, a, alpha, b):
    return a * N**(-alpha) + b

In [ ]:
for name in ['textile', 'microplastics', 'pcuk', 'pollen', 'milk', 'bacteria_test', 'mlrod']:

    with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "rb") as f:
        data = pickle.load(f)
    
    raw_sizes = data["raw_sizes"]
    raw_acc = data["raw_acc"]
    aug_sizes = data["aug_sizes"]
    aug_acc = data["aug_acc"]


    idx = np.argsort(raw_sizes)
    
    cut = 0
    N_list = np.array(raw_sizes)[idx][cut:]
    raw_error = 1 - raw_acc[0, idx][cut:]
    aug_error = 1 - aug_acc[0, idx][cut:]
    
    raw_std = raw_acc[1, idx][cut:] + 1e-5
    aug_std = aug_acc[1, idx][cut:] + 1e-5
    
    raw_params, raw_covariance = curve_fit(
        scaling_law,
        N_list,
        raw_error,
        # sigma=raw_std,
        # absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )
    
    aug_params, aug_covariance = curve_fit(
        scaling_law,
        N_list,
        aug_error,
        # sigma=aug_std,
        # absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )
    
    a_raw, alpha_raw, b_raw = raw_params
    a_aug, alpha_aug, b_aug = aug_params
    
    plt.plot(
        N_list, raw_error,
        'o', color='tab:orange', label="Raw data"
    )
    
    plt.plot(
        N_list, scaling_law(N_list, *raw_params),
        '--', color='tab:orange',
        label=(
            rf"Fit: $a={a_raw:.2f}$, "
            rf"$\alpha={alpha_raw:.2f}$, "
            rf"$b={b_raw:.3f}$"
        )
    )
    
    plt.plot(
        N_list, aug_error,
        'o', color='tab:blue', label="Augmented data"
    )
    
    plt.plot(
        N_list, scaling_law(N_list, *aug_params),
        '--', color='tab:blue',
        label=(
            rf"Fit: $a={a_aug:.2f}$ , "
            rf"$\alpha={alpha_aug:.2f}$, "
            rf"$b={b_aug:.3f}$"
        )
    )
    
    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Number of training samples per class $N$")
    plt.ylabel("Error ($1-$Accuracy)")
    plt.title('PCUK')
    plt.legend(fontsize=10)
    plt.title(name)
    plt.show()

In [ ]:
common_N0 = []
for name in ['textile', 'microplastics', 'pcuk', 'pollen', 'milk', 'bacteria_test']:

    with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "rb") as f:
        data = pickle.load(f)
    common_N0.append( max(data["raw_sizes"]))
common_N0 = min(common_N0)

for name in ['textile', 'microplastics', 'pcuk', 'pollen', 'milk', 'bacteria_test']:

    with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "rb") as f:
        data = pickle.load(f)
    
    raw_sizes = data["raw_sizes"]
    raw_acc = data["raw_acc"]
    aug_sizes = data["aug_sizes"]
    aug_acc = data["aug_acc"]
    

    idx = np.argsort(raw_sizes)
    
    cut = 0
    N_list = np.array(raw_sizes)[idx][cut:]
    raw_error = 1 - raw_acc[0, idx][cut:]
    aug_error = 1 - aug_acc[0, idx][cut:]
    
    raw_std = raw_acc[1, idx][cut:] + 1e-5
    aug_std = aug_acc[1, idx][cut:] + 1e-5
    
    raw_params, raw_covariance = curve_fit(
        scaling_law,
        N_list,
        raw_error,
        sigma=raw_std,
        absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )
    
    aug_params, aug_covariance = curve_fit(
        scaling_law,
        N_list,
        aug_error,
        sigma=aug_std,
        absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )
    
    a_raw, alpha_raw, b_raw = raw_params
    a_aug, alpha_aug, b_aug = aug_params
    
    raw_normalized_error = (scaling_law(N_list, *raw_params) - b_raw)/(scaling_law([common_N0], *raw_params) - b_raw)
    aug_normalized_error = (scaling_law(N_list, *aug_params) - b_aug)/(scaling_law([common_N0], *aug_params) - b_aug)

    
    plt.plot(
        N_list/common_N0, raw_normalized_error,
        '--', #color='tab:orange',
        label=(rf"{name}")
    )
    plt.xscale("log")
    plt.yscale("log")
    
    plt.xlabel("Training samples per class ratio $N/N_0$")
    plt.ylabel("$1-$Accuracy")
    plt.legend(fontsize=10)
    plt.title('Aug spectra')

In [ ]:
names = ['textile', 'microplastics', 'pollen', 'pcuk', 'milk', 'bacteria_test']

scaling_diff = {}

for name in names:

    with open(f"../outputs/processed_results/{name}_scaling_results.pkl", "rb") as f:
        data = pickle.load(f)
    
    raw_sizes = data["raw_sizes"]
    raw_acc = data["raw_acc"]
    aug_sizes = data["aug_sizes"]
    aug_acc = data["aug_acc"]
    

    idx = np.argsort(raw_sizes)
    N_list = np.array(raw_sizes)[idx]
    raw_error = 1 - raw_acc[0, idx]
    aug_error = 1 - aug_acc[0, idx]
    
    raw_std = raw_acc[1, idx] + 1e-5
    aug_std = aug_acc[1, idx] + 1e-5
    
    raw_params, raw_covariance = curve_fit(
        scaling_law,
        N_list,
        raw_error,
        sigma=raw_std,
        absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )
    
    aug_params, aug_covariance = curve_fit(
        scaling_law,
        N_list,
        aug_error,
        sigma=aug_std,
        absolute_sigma=True,
        p0=[0.5, 0.3, 0.05],
        bounds=([0, 0, 0], [np.inf, 5, 1])
    )

    scaling_diff[name] = aug_params[1] - raw_params[1]
    
plt.barh(scaling_diff.keys(), scaling_diff.values())

# Complexity metrics

In [ ]:
def pca_intrinsic_dimension(X, variance_threshold=0.95):
    Xc = X - np.mean(X, axis=0)
    pca = PCA()
    pca.fit(Xc)
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    dim = np.argmax(cumulative >= variance_threshold) + 1

    return { "intrinsic_dimension": dim,
            "explained_variance": cumulative[dim-1],
            "variance_curve": cumulative}

def spectral_entropy(X):
    spectrum = np.mean(np.abs(X), axis=0)
    p = spectrum / np.sum(spectrum)
    
    return entropy(p)

def participation_ratio(X):
    Xc = X - np.mean(X, axis=0)
    cov = np.cov(Xc, rowvar=False)
    eigenvalues = np.linalg.eigvalsh(cov)
    eigenvalues = eigenvalues[eigenvalues > 0]
    PR = (
        np.sum(eigenvalues)**2 /
        np.sum(eigenvalues**2)
    )

    return PR

def spectral_roughness(X):
    derivative = np.diff(X, axis=1)
    roughness = np.mean(
        derivative**2
    )

    return roughness

def fisher_ratio(X, y):
    classes = np.unique(y)
    global_mean = np.mean(X, axis=0)
    between = 0
    within = 0

    for c in classes:
        Xc = X[y == c]
        mean_c = np.mean(Xc, axis=0)
        between += len(Xc) * np.sum(
            (mean_c-global_mean)**2
        )
        within += np.sum(
            (Xc-mean_c)**2
        )

    return between / within

def spectral_complexity_report(X, y=None):
    results = {}
    pca = pca_intrinsic_dimension(X)
    results["intrinsic_dimension"] = (
        pca["intrinsic_dimension"]
    )

    results["spectral_entropy"] = (
        spectral_entropy(X)
    )

    results["participation_ratio"] = (
        participation_ratio(X)
    )

    results["spectral_roughness"] = (
        spectral_roughness(X)
    )

    if y is not None:
        results["fisher_ratio"] = (
            fisher_ratio(X,y)
        )

    return results

In [ ]:
from src.data.sampling import get_training_data_spectra, get_training_data_hyperspectral
all_complexity_metrics = {}


In [ ]:

zarr_path = "/mnt/ssd3/eirik/ProcessedData/milk.zarr"
labels, spectra, wn, _ = get_training_data_spectra(split = None,
                                                   zarr_path = zarr_path,
                                                   spectra_per_class = int(2**15)
                                                  )


metrics = spectral_complexity_report(
    spectra,
    labels
)

print('\n')
for k,v in metrics.items():
    print(f"{k:25s}: {v:.6f}")
    
all_complexity_metrics['milk'] = metrics


In [ ]:

zarr_path = "/mnt/ssd3/eirik/ProcessedData/ase_pollen_library.zarr"
labels, spectra, wn, _ = get_training_data_hyperspectral(split = None,
                                                         zarr_path = zarr_path,
                                                         spectra_per_class =int(2**10),
                                                         patch_size = 64,
                                                         background_max = 0.05,
                                                         sample_min = 0.3)


metrics = spectral_complexity_report(
    spectra,
    labels
)

print('\n')
for k,v in metrics.items():
    print(f"{k:25s}: {v:.6f}")
    
all_complexity_metrics['pollen'] = metrics


In [ ]:

zarr_path = "/mnt/ssd3/eirik/ProcessedData/milk.zarr"
labels, spectra, wn, _ = get_training_data_spectra(split = None,
                                                   zarr_path = zarr_path,
                                                   spectra_per_class = int(2**15)
                                                  )


metrics = spectral_complexity_report(
    spectra,
    labels
)

print('\n')
for k,v in metrics.items():
    print(f"{k:25s}: {v:.6f}")
    
all_complexity_metrics['milk'] = metrics


In [ ]:
zarr_path = "/mnt/ssd3/eirik/ProcessedData/pcuk.zarr"
labels, spectra, wn, _ = get_training_data_spectra(split = None,
                                                   zarr_path = zarr_path,
                                                   spectra_per_class = int(2**11),
                                                   classes = ["Normal epithelium", "Normal stroma", 
                                                              "Cancerous epithelium", "Cancer-associated stroma"]
                                                  )


metrics = spectral_complexity_report(
    spectra,
    labels
)

print('\n')
for k,v in metrics.items():
    print(f"{k:25s}: {v:.6f}")
all_complexity_metrics['pcuk'] = metrics


In [ ]:
zarr_path = "/mnt/ssd3/eirik/ProcessedData/microplastics_library.zarr"
labels, spectra, wn, _ = get_training_data_hyperspectral(split = None,
                                                         zarr_path = zarr_path,
                                                         spectra_per_class =int(2**10),
                                                         patch_size = 128,
                                                         background_max = 0.1,
                                                         sample_min = 0.5)


metrics = spectral_complexity_report(
    spectra,
    labels
)

print('\n')
for k,v in metrics.items():
    print(f"{k:25s}: {v:.6f}")

all_complexity_metrics['microplastics'] = metrics


In [ ]:
all_complexity_metrics

# Performance for different augmentations

In [ ]:
acc_list_16 = []
std_list_16 = []

acc_list_16.append(raw_acc[:, np.array(raw_sizes) == 16][0,0])
std_list_16.append(raw_acc[:, np.array(raw_sizes) == 16][1,0])

acc_list_16.append(noise_acc[:, np.array(noise_sizes) == 16][0,0])
std_list_16.append(noise_acc[:, np.array(noise_sizes) == 16][1,0])

acc_list_16.append(aug_acc[:, np.array(aug_sizes) == 16][0,0])
std_list_16.append(aug_acc[:, np.array(aug_sizes) == 16][1,0])

In [ ]:
x_labels = ['raw', 'noise', 'mie'] 
x = range(len(acc_list_16))
plt.step(x, acc_list_16, where='post', label='Mean Accuracy', color='tab:blue', alpha=0.6)
plt.errorbar(x , acc_list_16, yerr=np.array(std_list_16), fmt='o', capsize=5)

plt.xticks(x, x_labels)

plt.xlabel('Data', fontsize=12, labelpad=15)
plt.ylabel('Accuracy', fontsize=12, labelpad=15)
plt.show()

# Confusion Matrix

In [ ]:
img_result = load_image_outputs(store, trial_id, image_name)
result_dict[128]['N128_seed00']

In [ ]:
gt = result_dict[128]['N128_seed00']['gt']
pred = result_dict[128]['N128_seed00']['pred']  

cm = confusion_matrix(gt, pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

class_labels = ['ABS', 'PA12', 'PEHD', 'PET', 'PHB', 'PS', 'PVAC', 'PVC']       # MUST FIX THIS!

fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(cm_normalized, cmap='Blues')

ax.set_xticks(np.arange(len(class_labels)))
ax.set_yticks(np.arange(len(class_labels)))
ax.set_xticklabels(class_labels, rotation=0, ha='center', fontsize=14)
ax.set_yticklabels(class_labels, fontsize=14)

ax.xaxis.set_ticks_position('bottom')
ax.xaxis.set_label_position('bottom')

for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        ax.text(
            j, i,
            f"{cm_normalized[i, j]:.1f}%",
            ha='center',
            va='center',
            color='white' if cm_normalized[i, j] > 50 else 'black', 
            fontsize=10
        )
    
    
ax.set_xlabel('Predicted', fontsize=22, labelpad=20)
ax.set_ylabel('Ground Truth', fontsize=22, labelpad=20)
plt.tight_layout()
plt.show()